In [1]:

import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin


class LassoReg(RegressorMixin, BaseEstimator):
    def __init__(
        self,
        learning_rate=0.01,
        epochs=5000,
        alpha=0.1
    ):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.alpha = alpha

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)

        if X.ndim != 2:
            raise ValueError("X must be a 2D array.")
        if len(X) != len(y) or len(y) == 0:
            raise ValueError("X and y must have equal, nonzero lengths.")
        if not np.isfinite(X).all() or not np.isfinite(y).all():
            raise ValueError("X and y must contain only finite values.")
        if self.alpha < 0:
            raise ValueError("alpha must be non-negative.")
        if self.learning_rate <= 0:
            raise ValueError("learning_rate must be positive.")
        if self.epochs < 1:
            raise ValueError("epochs must be at least 1.")

        n_samples, n_features = X.shape
        self.n_features_in_ = n_features

        self.w = np.zeros(n_features)
        self.b = 0.0
        self.loss_history = []

        for epoch in range(self.epochs):
            # Forward pass
            y_pred = X @ self.w + self.b
            error = y_pred - y

            # MSE gradient + L1 subgradient
            dw = (2 / n_samples) * (X.T @ error)
            dw += (self.alpha / n_samples) * np.sign(self.w)

            # Intercept is not regularized
            db = 2 * np.mean(error)

            # Gradient descent update
            self.w -= self.learning_rate * dw
            self.b -= self.learning_rate * db

            # Track objective
            updated_pred = X @ self.w + self.b
            mse = np.mean((updated_pred - y) ** 2)
            penalty = (
                self.alpha / n_samples
            ) * np.sum(np.abs(self.w))

            self.loss_history.append(mse + penalty)

        self.coef_ = self.w.copy()
        self.intercept_ = self.b

        return self

    def predict(self, X):
        if not hasattr(self, "w"):
            raise ValueError("Fit the model before predicting.")

        X = np.asarray(X, dtype=float)

        if X.ndim != 2:
            raise ValueError("X must be a 2D array.")
        if X.shape[1] != self.n_features_in_:
            raise ValueError("X has an incorrect number of features.")

        return X @ self.w + self.b


In [2]:

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Lasso, Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load data
data = load_diabetes()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define pipelines
models = {
    "Scratch Lasso": make_pipeline(
        StandardScaler(),
        LassoReg(
            learning_rate=0.01,
            epochs=5000,
            alpha=10.0
        )
    ),
    "Sklearn Lasso": make_pipeline(
        StandardScaler(),
        Lasso(alpha=5.0, max_iter=10000)
    ),
    "Ridge": make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    ),
    "Linear Regression": make_pipeline(
        StandardScaler(),
        LinearRegression()
    )
}

# Train and evaluate
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    print(f"\n{name}")
    print(f"MAE:  {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²:   {r2:.4f}")



Scratch Lasso
MAE:  42.829
RMSE: 53.751
R²:   0.4547

Sklearn Lasso
MAE:  43.513
RMSE: 53.264
R²:   0.4645

Ridge
MAE:  42.812
RMSE: 53.777
R²:   0.4541

Linear Regression
MAE:  42.794
RMSE: 53.853
R²:   0.4526
